# SPARK ETL - BẢNG PHỤ, TỐI ƯU & BENCHMARK

## PHẦN 1: KẾT NỐI SPARK VÀ HDFS

In [1]:
from pyspark.sql import SparkSession, functions as F, Window

BASE = "hdfs://namenode:8020"
spark = (SparkSession.builder.appName("giai-doan-4")
         .config("spark.hadoop.fs.defaultFS", BASE)
         .config("spark.sql.warehouse.dir", f"{BASE}/user/hive/warehouse")
         .config("hive.metastore.uris", "thrift://hive-metastore:9083")
         .config("spark.sql.parquet.writeLegacyFormat", "true")   # de Hive 2.3 doc duoc
         .config("spark.driver.memory", "3g")
         .config("spark.sql.shuffle.partitions", "16")
         .config("spark.hadoop.dfs.replication", "1")
         .config("spark.local.dir", "/tmp/spark")
         .enableHiveSupport().getOrCreate())

spark.sql("CREATE DATABASE IF NOT EXISTS fifa")
# Kiểm tra phiên bản Spark xem đã kết nối thành công chưa
print("Spark version:", spark.version)

Spark version: 3.5.0


In [2]:
print(spark.conf.get("spark.hadoop.dfs.replication", "chưa đặt"))
print(spark.conf.get("spark.local.dir"))

1
/tmp/spark


## PHẦN 2: ĐỌC VÀ KIỂM TRA DỮ LIỆU TỪ HDFS
### Bước 1: Đọc dữ liệu cầu thủ từ HDFS vào Spark DataFrame

In [3]:
# Đọc file male_players.csv từ HDFS
df_male = spark.read.option("header", "true") \
                    .option("inferSchema", "true") \
                    .csv("hdfs://namenode:8020/fifa/raw/players/gender=male/male_players.csv")

# Kiểm tra số lượng dòng và hiển thị 5 dòng đầu tiên
print("Tổng số cầu thủ nam:", df_male.count())
df_male.show(5)

Tổng số cầu thủ nam: 10003590
+---------+--------------------+------------+-----------+----------------+--------------+--------------------+----------------+-------+---------+---------+--------+---+----------+---------+---------+---------+--------------+------------+------------+-------------------+-------------+------------------+----------------+----------------+------------------------------+--------------+----------------+--------------+---------------+--------------------+--------------+---------+-----------+------------------------+-------------+----------------+---------+------------------+--------------------+--------------------+----+--------+-------+---------+---------+------+------------------+-------------------+--------------------------+-----------------------+-----------------+---------------+-----------+-----------------+------------------+------------------+---------------------+---------------------+----------------+------------------+----------------+----------------

In [4]:
#Chuyển sang dạng xem theo chiều dọc (dễ nhìn toàn bộ chi tiết của 1-2 cầu thủ)
df_male.show(2, vertical=True, truncate=False)

-RECORD 0----------------------------------------------------------------------------------------------------------------------------------------
 player_id                      | 158023                                                                                                         
 player_url                     | /player/158023/lionel-messi/230009                                                                             
 fifa_version                   | 23                                                                                                             
 fifa_update                    | 9                                                                                                              
 fifa_update_date               | 2023-01-13                                                                                                     
 short_name                     | L. Messi                                                                                  

### Bước 2: Kiểm tra cấu trúc, số lượng và dữ liệu mẫu

In [5]:
# Kiểm tra cấu trúc dữ liệu
df_male.printSchema()

# Kiểm tra các cột cần thiết cho ETL
columns_to_check = [
    "player_id", "short_name", "club_name",
    "nationality_name", "player_positions",
    "age", "overall", "potential",
    "wage_eur", "value_eur"
]

for c in columns_to_check:
    if c in df_male.columns:
        print(f"{c}: có trong dữ liệu")
    else:
        print(f"{c}: KHÔNG có trong dữ liệu")

root
 |-- player_id: integer (nullable = true)
 |-- player_url: string (nullable = true)
 |-- fifa_version: integer (nullable = true)
 |-- fifa_update: integer (nullable = true)
 |-- fifa_update_date: date (nullable = true)
 |-- short_name: string (nullable = true)
 |-- long_name: string (nullable = true)
 |-- player_positions: string (nullable = true)
 |-- overall: integer (nullable = true)
 |-- potential: integer (nullable = true)
 |-- value_eur: integer (nullable = true)
 |-- wage_eur: integer (nullable = true)
 |-- age: integer (nullable = true)
 |-- dob: date (nullable = true)
 |-- height_cm: integer (nullable = true)
 |-- weight_kg: integer (nullable = true)
 |-- league_id: integer (nullable = true)
 |-- league_name: string (nullable = true)
 |-- league_level: integer (nullable = true)
 |-- club_team_id: integer (nullable = true)
 |-- club_name: string (nullable = true)
 |-- club_position: string (nullable = true)
 |-- club_jersey_number: integer (nullable = true)
 |-- club_loane

## PHẦN 3: LÀM SẠCH DỮ LIỆU VÀ CHUẨN HÓA DỮ LIỆU BẰNG SPARK
### Bước 1: Kiểm tra chất lượng dữ liệu khi làm sạch
Kiểm tra kiểu dữ liệu, giá trị thiếu và các trường thông tin quan trọng để xác định những vấn đề cần xử lý trước khi thực hiện ETL.

In [6]:
from pyspark.sql import functions as F
from pyspark.sql.types import StringType
# 1. Kiểm tra kiểu dữ liệu
df_male.printSchema()

# 2. Kiểm tra giá trị thiếu ở các cột quan trọng
key_cols = [
    c for c in [
        "player_id",
        "short_name",
        "club_name",
        "nationality_name",
        "player_positions",
        "age",
        "overall",
        "potential",
        "wage_eur",
        "value_eur"
    ]
    if c in df_male.columns
]
missing_stats = df_male.agg(*[
    F.sum(
        F.when(F.col(c).isNull(), 1).otherwise(0)
    ).alias(c)
    for c in key_cols
])
print("Số lượng giá trị NULL ở các cột quan trọng:")
missing_stats.show(truncate=False)

root
 |-- player_id: integer (nullable = true)
 |-- player_url: string (nullable = true)
 |-- fifa_version: integer (nullable = true)
 |-- fifa_update: integer (nullable = true)
 |-- fifa_update_date: date (nullable = true)
 |-- short_name: string (nullable = true)
 |-- long_name: string (nullable = true)
 |-- player_positions: string (nullable = true)
 |-- overall: integer (nullable = true)
 |-- potential: integer (nullable = true)
 |-- value_eur: integer (nullable = true)
 |-- wage_eur: integer (nullable = true)
 |-- age: integer (nullable = true)
 |-- dob: date (nullable = true)
 |-- height_cm: integer (nullable = true)
 |-- weight_kg: integer (nullable = true)
 |-- league_id: integer (nullable = true)
 |-- league_name: string (nullable = true)
 |-- league_level: integer (nullable = true)
 |-- club_team_id: integer (nullable = true)
 |-- club_name: string (nullable = true)
 |-- club_position: string (nullable = true)
 |-- club_jersey_number: integer (nullable = true)
 |-- club_loane

### *Kết quả kiểm tra giá trị thiếu trong dữ liệu*

Sau khi thực hiện kiểm tra dữ liệu bằng Apache Spark, kết quả thống kê giá trị NULL trên 10 thuộc tính quan trọng của bộ dữ liệu cầu thủ nam FIFA cho thấy phần lớn các thuộc tính không có giá trị thiếu. Cụ thể, các cột `player_id`, `short_name`, `nationality_name`, `player_positions`, `age`, `overall` và `potential` đều có 0 giá trị NULL.

Tuy nhiên, một số thuộc tính vẫn tồn tại dữ liệu thiếu, bao gồm:

- **`club_name`:** có 116.395 giá trị NULL.
- **`wage_eur`:** có 115.974 giá trị NULL.
- **`value_eur`:** có 132.797 giá trị NULL.

Trong đó, `value_eur` là thuộc tính có số lượng giá trị thiếu lớn nhất trong các cột được kiểm tra. Những giá trị này có thể ảnh hưởng đến các phép thống kê về giá trị thị trường, tiền lương và phân tích theo câu lạc bộ nếu không được xem xé bằng Spark.

>**Kết luận:** Kết quả kiểm tra cho thấy dữ liệu có chất lượng tương đối tốt ở các thuộc tính định danh cầu thủ, quốc tịch, vị trí thi đấu, tuổi và các chỉ số năng lực. Tuy nhiên, vẫn tồn tại giá trị thiếu ở các thuộc tính liên quan đến câu lạc bộ và tài chính. Vì vậy, cần tiếp tục xử lý và chuẩn hóa dữ liệu ở bước tiếp theo trước khi thực hiện các phép tổng hợp, xây dựng bảng dữ liệu phụ và benchmark bằng Spark.

### Bước 2: Chuẩn hóa chuỗi và xử lý giá trị thiếu
Chuẩn hóa các cột kiểu chuỗi bằng cách loại bỏ khoảng trắng thừa ở đầu và cuối, đồng thời chuyển các chuỗi rỗng hoặc giá trị thiếu được biểu diễn bằng văn bản thành giá trị NULL thực sự.

In [7]:
from pyspark.sql import functions as F
from pyspark.sql.types import StringType

# Cột dùng ở các bảng phụ + cột phiên bản (nếu có) để khử trùng đúng
version_cols = [c for c in ["fifa_version", "fifa_update"] if c in df_male.columns]
keep_cols = [c for c in columns_to_check + version_cols if c in df_male.columns]
print("Các cột phiên bản tìm thấy:", version_cols)
print("Các cột được giữ lại:", keep_cols)

df_sel = df_male.select(*keep_cols)
string_cols = [f.name for f in df_sel.schema.fields
               if isinstance(f.dataType, StringType)]

def clean_str(c):
    v = F.trim(F.col(c))
    return (F.when(v.isNull() | (v == "")
                   | F.lower(v).isin("null", "none", "n/a", "na"),
                   F.lit(None).cast("string"))
             .otherwise(v).alias(c))

df_male_clean = df_sel.select(
    *[clean_str(c) if c in string_cols else F.col(c) for c in df_sel.columns]
)
print("Số cột kiểu chuỗi đã chuẩn hóa:", len(string_cols))
print("Đã hoàn thành bước chuẩn hóa chuỗi. Số cột còn lại:", len(df_male_clean.columns))

Các cột phiên bản tìm thấy: ['fifa_version', 'fifa_update']
Các cột được giữ lại: ['player_id', 'short_name', 'club_name', 'nationality_name', 'player_positions', 'age', 'overall', 'potential', 'wage_eur', 'value_eur', 'fifa_version', 'fifa_update']
Số cột kiểu chuỗi đã chuẩn hóa: 4
Đã hoàn thành bước chuẩn hóa chuỗi. Số cột còn lại: 12


**Đoạn code này xử lý các vấn đề sau:**

- Chỉ giữ lại **12 cột cần dùng** cho các bảng phụ (10 cột chính và 2 cột phiên bản `fifa_version`, `fifa_update`), giúp giảm dung lượng xử lý và dữ liệu tạm trên đĩa.
- Cắt khoảng trắng thừa ở đầu và cuối chuỗi: `" Paris Saint Germain "` → `"Paris Saint Germain"`.
- Chuỗi rỗng `""` → NULL.
- Chuỗi `"N/A"`, `"na"`, `"none"` hoặc `"null"` (không phân biệt hoa thường) → NULL.

Các giá trị thiếu được giữ lại dưới dạng NULL, chưa xoá hàng loạt. Đây là lựa chọn an toàn hơn với dữ liệu FIFA, vì một cầu thủ có thể thiếu câu lạc bộ hoặc lương nhưng vẫn có các thông tin khác hữu ích cho phân tích.

### Bước 3: Kiểm tra trùng lặp theo bộ khoá (`player_id`, `fifa_version`, `fifa_update`)

Mỗi bản ghi trong dữ liệu đại diện cho **một cầu thủ ở một phiên bản FIFA cụ thể**, nên một cầu thủ có thể xuất hiện nhiều lần hợp lệ qua các phiên bản. Vì vậy:

- **Không** loại trùng chỉ dựa trên `player_id`, vì sẽ xoá nhầm lịch sử của cùng một cầu thủ ở các phiên bản khác nhau.
- Bộ khoá để xác định một bản ghi là duy nhất gồm 3 cột: `player_id`, `fifa_version`, `fifa_update`.

Trước khi loại bỏ, ta **đếm số nhóm bị trùng theo bộ khoá này**. Cách này chỉ xáo trộn 3 cột nên nhẹ hơn nhiều so với `dropDuplicates()` trên toàn bộ cột, giúp hạn chế dữ liệu tạm ghi xuốt bản ghi. một bản ghi.

In [8]:
keys = ["player_id", "fifa_version", "fifa_update"]

dup_groups = (df_male_clean
              .groupBy(*keys).count()
              .filter(F.col("count") > 1)
              .count())
print("Số nhóm bị trùng theo khoá:", dup_groups)

Số nhóm bị trùng theo khoá: 0


### Bước 4: Chọn theo kết quả
- Nếu kết quả bằng 0: dữ liệu không có bản ghi trùng, không cần khử trùng.
- Nếu kết quả lớn hơn 0: dùng `dropDuplicates` theo bộ khoá trên, mỗi cầu thủ ở mỗi phiên bản chỉ giữ một bản ghi.

In [9]:
df_male_clean = df_male_clean.dropDuplicates(keys)
print("Đã loại bỏ bản ghi trùng lặp theo khoá:", keys)

Đã loại bỏ bản ghi trùng lặp theo khoá: ['player_id', 'fifa_version', 'fifa_update']


### Bước 5: Kiểm tra kết quả sau làm sạch

Kiểm tra số lượng bản ghi, cấu trúc dữ liệu và các giá trị thiếu sau khi chuẩn hóa. So sánh kết quả với dữ liệu ban đầu để đánh giá những thay đổi sau quá trình làm sạch.

In [10]:
from pyspark.sql import functions as F

# Chỉ lấy các cột quan trọng thực sự có trong dữ liệu đã làm sạch
key_cols = [c for c in columns_to_check if c in df_male_clean.columns]

# 1. Số dòng trước làm sạch (chỉ quét file, không xáo trộn nên rất nhẹ)
rows_before = df_male.count()

# 2. Số dòng sau làm sạch + đếm NULL: gộp thành MỘT lần chạy duy nhất
stats = df_male_clean.agg(
    F.count(F.lit(1)).alias("_rows"),
    *[F.sum(F.when(F.col(c).isNull(), 1).otherwise(0)).alias(c) for c in key_cols]
).collect()[0].asDict()

rows_after = stats.pop("_rows")

print(f"Số dòng trước làm sạch: {rows_before:,}")
print(f"Số dòng sau làm sạch:   {rows_after:,}")
print(f"Số dòng chênh lệch:      {rows_before - rows_after:,}")

# 3. Cấu trúc dữ liệu (chỉ đọc metadata, không tốn tài nguyên)
df_male_clean.printSchema()

# 4. Số lượng NULL ở các cột quan trọng
import pandas as pd
display(pd.DataFrame([stats]))

Số dòng trước làm sạch: 10,003,590
Số dòng sau làm sạch:   10,003,590
Số dòng chênh lệch:      0
root
 |-- player_id: integer (nullable = true)
 |-- short_name: string (nullable = true)
 |-- club_name: string (nullable = true)
 |-- nationality_name: string (nullable = true)
 |-- player_positions: string (nullable = true)
 |-- age: integer (nullable = true)
 |-- overall: integer (nullable = true)
 |-- potential: integer (nullable = true)
 |-- wage_eur: integer (nullable = true)
 |-- value_eur: integer (nullable = true)
 |-- fifa_version: integer (nullable = true)
 |-- fifa_update: integer (nullable = true)



,player_id,short_name,club_name,nationality_name,player_positions,age,overall,potential,wage_eur,value_eur
0,0,0,116395,0,0,0,0,0,115974,132797


In [11]:
# Chạy thử vài dòng
df_male_clean.limit(5).show(truncate=False)

+---------+-----------+--------------+----------------+----------------+---+-------+---------+--------+---------+------------+-----------+
|player_id|short_name |club_name     |nationality_name|player_positions|age|overall|potential|wage_eur|value_eur|fifa_version|fifa_update|
+---------+-----------+--------------+----------------+----------------+---+-------+---------+--------+---------+------------+-----------+
|2        |G. Pasquale|Udinese Calcio|Italy           |LM, LB          |32 |68     |68       |9000    |675000   |15          |1          |
|2        |G. Pasquale|Udinese       |Italy           |LM              |32 |68     |68       |9000    |675000   |15          |18         |
|2        |G. Pasquale|Udinese       |Italy           |LM              |32 |68     |68       |9000    |675000   |15          |28         |
|2        |G. Pasquale|Udinese       |Italy           |LM              |33 |69     |69       |7000    |625000   |16          |19         |
|2        |G. Pasquale|Udin

### Kết quả kiểm tra dữ liệu sau làm sạch
Sau khi chọn lọc các cột cần dùng, chuẩn hóa chuỗi và loại bỏ bản ghi trùng lặp theo bộ khóa (**`player_id`**, **`fifa_version`**, **`fifa_update`**) bằng Apache Spark, dữ liệu được kiểm tra lại về số lượng bản ghi, cấu trúc các thuộc tính và số giá tri thiếu

**1. Kết quả so sánh số lượng bản ghi**
- Số dòng trước khi làm sạch: **10.003.590 dòng**
- Số dòng sau khi làm sạch: **10.003.590 dòng**
- Số dòng chênh lệch: **0 dòng**

> Số dòng không thay đổi nghĩa là không có bản ghi nào trùng nhau theo bộ khóa**`player_id`**, **`fifa_version`**, **`fifa_update`**. Mỗi bản ghi là một cầu thủ ở một phiên bản FIFA cụ thể, nên việc một cầu thủ xuất hiện nhiều lần qua các phiên bản là hợp lệ và được giữ nguyên. Bước chuẩn hóa chuỗi chỉ thay đổi giá trị bên trong bản ghi, không làm thay đổi số dòng.

**2. Kết quả kiểm tra giá trị thiếu (NULL)**
Kết quả `printSchema()` cho thấy dữ liệu sau làm sạch gồm **12 thuộc tính**:
- Kiểu số nguyên (`interger`): `player_id`, `age`, `overall`, `potential`, `wage_eur`, `valuje_eur`, `fifa_version`, `fifa_update`.
- Kiểu chuỗi (`string`): `short_name`, `club_name `, `nationality_name`, `player_positions`.

**3. Kết quả kiểm tra giá trị thiếu (NULL)**
|Thuộc tính|Số giá trị NULL|Tỷ lệ so với tổng số dòng|
|---|---|---
|`player_id`. `short_name`, `nationality_name`, `player_positions`, `age`, `overall`, `potential` | 0 | 0% |
|`club_name`|116.395|khoảng 1,16%|
|`wage_eur`|115.974|khoảng 1,16%$|
|`value_eur`|132.797|khoảng 1,33%|

Các thuộc tính dịnh danh và chỉ số chuyên môn đều đầy đủ. Giá trị thiếu chỉ tập trung ở `club_name`, `wage_eur` và `value_eur`, với tỷ lệ thấp (dưới 1,4%). Các giá trị này được giữ dưới dạng NULL, không xóa hàng loạt, vì cầu thủ thiếu câu lạc bộ hoặc lương vẫn có đầy đủ thông tin khác hữu ích cho phân tích.

**4. Kết luận**
- Quá trình làm sạch Spark đã chọn lọc 12 cột cần thiết, chuẩn hóa chuỗi và kiểm tra trùng lặp theo bộ khóa mà không làm thay đổi tổng số dòng (10.003.590).
- Cấu trúc dữ liệu rõ ràng, các kiểu dữ liệu phù hợp, tạo điều kiện thuận lợi cho các bước ETL tiếp theo như lọc cầu thủ và thống kê theo quốc tịch, câu lạc bộ, vị trí thi đấu.
- Các giá trị NULL ở `clunb_name`, `wage_eur`, `value_eur` cần được lưu ý khi tính toán thống kê. Các hàm tổng hợp của Spark như `avg` và `sum` tự bỏ qua giá trị NULL, nên không làm sai kết quả các bảng phụ.

## PHẦN 4: ETL VÀ XÂY DỰNG CÁC BẢNG DỮ LIỆU PHỤ

In [12]:
# Ghi dữ liệu sạch ra partque
df_male_clean.write.mode("overwrite").parquet(
    "hdfs://namenode:8020/fifa/tmp/male_clean"
)
df_male_clean = spark.read.parquet(
    "hdfs://namenode:8020/fifa/tmp/male_clean"
)

### Bước 1: Thực hiện các phép biến đổi ETL cơ bản trên Dataframe (df_male)
Bây giờ dữ liệu đã sẵn sàng trong biến df_male, bạn hãy tạo các ô code mới trong Jupyter để lọc, làm sạch và trích xuất thông tin theo yêu cầu của đồ án:
1. Lọc dữ liệu cầu thủ có chỉ số tiềm năng cao (Potential >= 85):

In [13]:
# Lọc các cầu thủ có tiềm năng từ 85 trở lên
top_potential_players = df_male_clean.filter(
    F.col("potential") >= 85
)

print(
    "Số lượng cầu thủ tiềm năng cao:",
    top_potential_players.count()
)

top_potential_players.select(
    "short_name",
    "club_name",
    "overall",
    "potential"
).show(5, truncate=False)

Số lượng cầu thủ tiềm năng cao: 189739
+----------+------------+-------+---------+
|short_name|club_name   |overall|potential|
+----------+------------+-------+---------+
|Iniesta   |FC Barcelona|89     |89       |
|Iniesta   |FC Barcelona|89     |89       |
|Iniesta   |FC Barcelona|89     |89       |
|Iniesta   |FC Barcelona|88     |88       |
|Iniesta   |FC Barcelona|88     |88       |
+----------+------------+-------+---------+
only showing top 5 rows



2. Tạo bảng

In [14]:
from pyspark.sql import functions as F, Window

# Phiên bản FIFA mới nhất trong dữ liệu
latest_version = df_male_clean.agg(F.max("fifa_version")).collect()[0][0]
print("Phiên bản mới nhất:", latest_version)

# Mỗi cầu thủ chỉ giữ lần cập nhật cuối cùng của phiên bản đó
w = Window.partitionBy("player_id").orderBy(F.col("fifa_update").desc())

df_latest = (
    df_male_clean
    .filter(F.col("fifa_version") == latest_version)
    .withColumn("rn", F.row_number().over(w))
    .filter(F.col("rn") == 1)
    .drop("rn")
    .persist()      # bảng rất nhỏ, giữ trong bộ nhớ cho các cell sau
)

print("Số cầu thủ (mỗi người một dòng):", df_latest.count())

Phiên bản mới nhất: 23
Số cầu thủ (mỗi người một dòng): 20621


3. Tính toán thống kê nhóm (Group By): Ví dụ tính số cầu thủ, tuổi trung bình, chỉ số overall trung bình và tổng giá trị thị trường (value_eur) theo từng quốc gia (nationality_name), trên dữ liệu phiên bản FIFA mới nhất (mỗi cầu thủ một bản ghi).

In [15]:
from pyspark.sql import functions as F

# 1. Tính toán thống kê nhóm và đặt tên alias
nationality_stats = (df_latest.groupBy("nationality_name")
    .agg(
        F.count("player_id").alias("total_players"),
        F.avg("overall").alias("avg_overall"),
        F.sum("value_eur").alias("total_value")
    )
    .orderBy(F.col("total_value").desc())
    .cache())

# 2. Hiển thị 10 quốc gia dẫn đầu
nationality_stats.show(10, truncate=False)

+----------------+-------------+------------------+-----------+
|nationality_name|total_players|avg_overall       |total_value|
+----------------+-------------+------------------+-----------+
|Spain           |1184         |68.59037162162163 |5265890000 |
|France          |1007         |67.22244289970209 |4572820000 |
|Brazil          |771          |71.22178988326849 |4308490000 |
|England         |1805         |63.282548476454295|4059420000 |
|Germany         |1274         |65.58084772370486 |3528040000 |
|Argentina       |1074         |68.1219739292365  |3167210000 |
|Italy           |711          |67.43881856540085 |2735925000 |
|Portugal        |404          |69.48267326732673 |2478050000 |
|Netherlands     |471          |67.18046709129511 |1976820000 |
|Belgium         |349          |66.63323782234957 |1263875000 |
+----------------+-------------+------------------+-----------+
only showing top 10 rows



### Bước 2: Tạo và lưu các bảng phụ (Auxiliary Tables)
Trong các yêu cầu của đồ án Big Data, việc tạo các bảng phụ tổng hợp (ví dụ: thống kê theo CLB, theo quốc tịch, hoặc theo vị trí) giúp tối ưu hóa truy vấn sau này. Bạn có thể lưu các bảng phụ này xuống định dạng Parquet (định dạng tối ưu nhất cho Spark) trên HDFS:

In [16]:
# BẢNG PHỤ 1: THỐNG KÊ THEO QUỐC TỊCH
nationality_stats = (
    df_latest
    .groupBy("nationality_name")
    .agg(
        F.count("player_id").alias("total_players"),
        F.avg("age").alias("avg_age"),
        F.avg("overall").alias("avg_overall"),
        F.sum("value_eur").alias("total_value")
    )
    .orderBy(F.col("total_value").desc())
)
nationality_stats.show(10, truncate=False)

+----------------+-------------+------------------+------------------+-----------+
|nationality_name|total_players|avg_age           |avg_overall       |total_value|
+----------------+-------------+------------------+------------------+-----------+
|Spain           |1184         |25.21114864864865 |68.59037162162163 |5265890000 |
|France          |1007         |24.43992055610725 |67.22244289970209 |4572820000 |
|Brazil          |771          |26.78988326848249 |71.22178988326849 |4308490000 |
|England         |1805         |24.065373961218835|63.282548476454295|4059420000 |
|Germany         |1274         |24.5447409733124  |65.58084772370486 |3528040000 |
|Argentina       |1074         |25.931098696461824|68.1219739292365  |3167210000 |
|Italy           |711          |25.873417721518987|67.43881856540085 |2735925000 |
|Portugal        |404          |24.730198019801982|69.48267326732673 |2478050000 |
|Netherlands     |471          |23.976645435244162|67.18046709129511 |1976820000 |
|Bel

### Bước 3: Tối ưu hóa (Optimization) & Benchmark
Caching / Persisting: Khi bạn thực hiện nhiều thao tác lặp đi lặp lại trên DataFrame lớn (như df_male), hãy dùng lệnh cache để giữ nó trên bộ nhớ RAM, giúp các câu lệnh chạy nhanh hơn ở những lần gọi sau:

In [17]:
print(spark)

In [18]:
# Lấy 5 dòng đầu tiên và hiển thị dưới dạng bảng trực quan
display(df_male_clean.limit(5).toPandas())

,player_id,short_name,club_name,nationality_name,player_positions,age,overall,potential,wage_eur,value_eur,fifa_version,fifa_update
0,2,G. Pasquale,Udinese,Italy,LM,32,68,68,9000,675000,15,14
1,2,G. Pasquale,Udinese,Italy,LM,32,68,68,9000,675000,15,19
2,2,G. Pasquale,Udinese,Italy,"LM, LB",32,69,69,10000,675000,15,38
3,2,G. Pasquale,Udinese,Italy,LM,33,69,69,7000,625000,16,33
4,2,G. Pasquale,Udinese,Italy,LM,33,69,69,7000,625000,16,36


In [19]:
# Benchmark thời gian chạy:
# Đo thời gian thực thi của truy vấn lớn bằng time.perf_counter().
# Kết quả giúp đánh giá hiệu suất xử lý dữ liệu của PySpark.
import time
from pyspark.sql.functions import avg

start = time.perf_counter()

df_male_clean.groupBy("club_name") \
    .agg(avg("wage_eur").alias("avg_wage")) \
    .show(10)

end = time.perf_counter()

print(f"Thời gian thực thi: {end - start:.2f} giây")

+---------------+------------------+
|      club_name|          avg_wage|
+---------------+------------------+
|  Coventry City| 5161.929317067035|
|  Leyton Orient| 3311.795218764096|
|   Grimsby Town| 2197.794481258036|
|       Alcorcón| 4800.777815353399|
|    Real Madrid|139740.27214823393|
|         Grödig| 3102.861562258314|
|AFC Bournemouth|26211.219663269327|
|      Feyenoord|14628.237560192616|
|         Chaves|5582.4492011788425|
|     PEC Zwolle| 5273.468407912425|
+---------------+------------------+
only showing top 10 rows

Thời gian thực thi: 2.24 giây


### Bước 4: Tạo bảng phụ theo câu lạc bộ
Bảng này giúp trả lời những câu hỏi như:
- Câu lạc bộ nào có tổng giá trị thị trường cầu thủ cao?
- Câu lạc bộ nào có mức lương trung bình cao?
- Chất lượng cầu thủ trung bình giữa các câu lạc bộ khác nhau thế nào?

In [20]:
# BẢNG PHỤ 2: Thống kê cầu thủ theo câu lạc bộ
club_stats = (
    df_latest
    .filter(F.col("club_name").isNotNull())      # bỏ nhóm không có câu lạc bộ
    .groupBy("club_name")
    .agg(
        F.count("player_id").alias("total_players"),
        F.avg("overall").alias("avg_overall"),
        F.avg("wage_eur").alias("avg_wage"),
        F.sum("value_eur").alias("total_value")
    )
    .orderBy(F.col("total_value").desc())
)
club_stats.show(10, truncate=False)

+-------------------+-------------+-----------------+------------------+-----------+
|club_name          |total_players|avg_overall      |avg_wage          |total_value|
+-------------------+-------------+-----------------+------------------+-----------+
|Manchester City    |26           |79.53846153846153|141076.92307692306|1255055000 |
|Liverpool          |59           |72.13559322033899|55795.76271186441 |1138045000 |
|Paris Saint Germain|26           |80.34615384615384|87826.92307692308 |1056575000 |
|Real Madrid        |33           |76.9090909090909 |129318.18181818182|1031050000 |
|FC Barcelona       |37           |75.91891891891892|100216.21621621621|1020625000 |
|FC Bayern München  |27           |78.44444444444444|61296.2962962963  |1004750000 |
|Chelsea            |35           |76.82857142857142|86142.85714285714 |959525000  |
|Inter              |28           |78.5             |60785.71428571428 |744025000  |
|Manchester United  |33           |76.06060606060606|85666.666666

### Bước 5: Tạo bảng phụ theo vị trí thi đấu
Lưu ý: player_positions có thể chứa nhiều vị trí trong cùng một ô, chẳng hạn RW, ST. Khi đó, kết quả đang thống kê theo chuỗi vị trí được ghi trong dữ liệu, chưa tách riêng từng vị trí.

In [21]:
# BẢNG PHỤ 3: Thống kê theo vị trí thi đấu
position_stats = (
    df_latest
    .groupBy("player_positions")
    .agg(
        F.count("player_id").alias("total_players"),
        F.avg("overall").alias("avg_overall"),
        F.avg("potential").alias("avg_potential"),
        F.avg("value_eur").alias("avg_value")
    )
    .orderBy(F.col("total_players").desc())
)
position_stats.show(10, truncate=False)

+----------------+-------------+------------------+-----------------+------------------+
|player_positions|total_players|avg_overall       |avg_potential    |avg_value         |
+----------------+-------------+------------------+-----------------+------------------+
|CB              |2679         |65.90182904068682 |70.73684210526316|2540618.1886508833|
|GK              |2311         |63.96841194288187 |69.44872349632193|1779923.2446576536|
|ST              |1937         |65.08569953536397 |70.21889519876098|2648788.351534061 |
|CDM, CM         |1050         |67.19047619047619 |71.34571428571428|3027390.0573613765|
|CM              |798          |61.728070175438596|69.7468671679198 |2554611.528822055 |
|CM, CDM         |762          |67.03937007874016 |71.47637795275591|3490732.1899736146|
|LB              |552          |64.84601449275362 |69.8713768115942 |2401047.3588342443|
|RB              |534          |64.82771535580524 |69.93258426966293|2250844.2776735458|
|CB, RB          |310

### Bước 6: Lưu các bảng phụ lên HDFS dưới dạng Parquet
Lưu 3 bảng phụ theo quốc tịch, câu lạc bộ và vị trí thi đấu

In [22]:
# Lưu 3 bảng phụ lên HDFS dưới dạng Parquet
nationality_stats.write.mode("overwrite").parquet("hdfs://namenode:8020/fifa/processed/nationality_summary")
club_stats.write.mode("overwrite").parquet("hdfs://namenode:8020/fifa/processed/club_summary")
position_stats.write.mode("overwrite").parquet("hdfs://namenode:8020/fifa/processed/position_summary")

print("Đã lưu 3 bảng phụ lên HDFS!")

Đã lưu 3 bảng phụ lên HDFS!


In [23]:
# Kiểm tra các thư mục đã lưu. Nếu kết quả đều là True, các thư mục tương ứng đang tồn tại trên HDFS.
spark._jsc.hadoopConfiguration().get("fs.defaultFS")

hdfs = spark._jvm.org.apache.hadoop.fs.FileSystem.get(
    spark._jsc.hadoopConfiguration()
)

for path in [
    "/fifa/processed/nationality_summary",
    "/fifa/processed/club_summary",
    "/fifa/processed/position_summary"
]:
    print(path, "->", hdfs.exists(
        spark._jvm.org.apache.hadoop.fs.Path(path)
    ))

/fifa/processed/nationality_summary -> True
/fifa/processed/club_summary -> True
/fifa/processed/position_summary -> True


## PHẦN 5: TỐI ƯU HÓA BẰNG CACHE VÀ BENCHMARK
### Bước 1: Đo thời gian truy vấn trước khi cache

In [24]:
import time

# Benchmark trước khi cache
start = time.perf_counter()

df_male_clean.groupBy("club_name").agg(
    F.avg("wage_eur").alias("avg_wage")
).show(10)

time_before_cache = time.perf_counter() - start

print(f"Thời gian trước cache: {time_before_cache:.2f} giây")

+---------------+------------------+
|      club_name|          avg_wage|
+---------------+------------------+
|  Coventry City| 5161.929317067035|
|  Leyton Orient| 3311.795218764096|
|   Grimsby Town| 2197.794481258036|
|       Alcorcón| 4800.777815353399|
|    Real Madrid|139740.27214823393|
|         Grödig| 3102.861562258314|
|AFC Bournemouth|26211.219663269327|
|      Feyenoord|14628.237560192616|
|         Chaves|5582.4492011788425|
|     PEC Zwolle| 5273.468407912425|
+---------------+------------------+
only showing top 10 rows

Thời gian trước cache: 1.28 giây


### Bước 2: Cache DataFrame và đo lại 
df_male.count() được dùng để kích hoạt việc tính toán và nạp DataFrame vào cache trước khi đo truy vấn. Vì vậy, thời gian nạp dữ liệu ban đầu không nằm trong thời gian benchmark thứ hai.

Lưu ý: Cache có thể làm truy vấn lặp lại nhanh hơn, nhưng không đảm bảo lúc nào cũng nhanh hơn. Kết quả còn phụ thuộc vào RAM, tài nguyên cụm, dữ liệu và tải hệ thống.

In [25]:
from pyspark import StorageLevel

# Lưu DataFrame vào bộ nhớ để tái sử dụng (không ghi xuống đĩa)
df_male_clean.persist(StorageLevel.MEMORY_ONLY)

# Kích hoạt việc đọc và lưu dữ liệu vào cache
df_male_clean.count()

# Benchmark sau khi cache
start = time.perf_counter()

df_male_clean.groupBy("club_name").agg(
    F.avg("wage_eur").alias("avg_wage")
).show(10)

time_after_cache = time.perf_counter() - start

print(f"Thời gian sau cache: {time_after_cache:.2f} giây")

+---------------+------------------+
|      club_name|          avg_wage|
+---------------+------------------+
|  Coventry City| 5161.929317067035|
|  Leyton Orient| 3311.795218764096|
|   Grimsby Town| 2197.794481258036|
|       Alcorcón| 4800.777815353399|
|    Real Madrid|139740.27214823393|
|         Grödig| 3102.861562258314|
|AFC Bournemouth|26211.219663269327|
|      Feyenoord|14628.237560192616|
|         Chaves|5582.4492011788425|
|     PEC Zwolle| 5273.468407912425|
+---------------+------------------+
only showing top 10 rows

Thời gian sau cache: 1.22 giây


### Bước 3: Tính mức cải thiện hiệu suất 
Nếu phần trăm dương, thời gian chạy giảm; nếu phần trăm âm, thời gian chạy tăng. Để kết quả đáng tin cậy hơn, bạn nên chạy mỗi trường hợp vài lần và so sánh các lần chạy tương ứng, vì lần chạy đầu có thể chịu ảnh hưởng của việc khởi tạo Spark và đọc dữ liệu từ HDFS.

In [26]:
import pandas as pd
from IPython.display import display

# Tính mức cải thiện hiệu suất
if time_before_cache > 0:
    improvement = (
        (time_before_cache - time_after_cache)
        / time_before_cache
    ) * 100

    # Tạo bảng kết quả benchmark
    benchmark_results = pd.DataFrame({
        "Chỉ tiêu": [
            "Thời gian trước khi cache",
            "Thời gian sau khi cache",
            "Mức cải thiện hiệu suất"
        ],
        "Giá trị": [
            f"{time_before_cache:.2f} giây",
            f"{time_after_cache:.2f} giây",
            f"{improvement:.2f}%"
        ]
    })

    display(benchmark_results)

# Giải phóng cache, tránh hiển thị DataFrame dài
_ = df_male_clean.unpersist()

print("Đã giải phóng cache thành công.")

,Chỉ tiêu,Giá trị
0,Thời gian trước khi cache,1.28 giây
1,Thời gian sau khi cache,1.22 giây
2,Mức cải thiện hiệu suất,5.02%


Đã giải phóng cache thành công.


## PHẦN 6: KIỂM TRA KẾT QUẢ VÀ TỔNG KẾT

In [27]:
# Kiểm tra đọc lại 3 bảng phụ từ HDFS

nationality_check = spark.read.parquet(
    "hdfs://namenode:8020/fifa/processed/nationality_summary"
)

club_check = spark.read.parquet(
    "hdfs://namenode:8020/fifa/processed/club_summary"
)

position_check = spark.read.parquet(
    "hdfs://namenode:8020/fifa/processed/position_summary"
)

# Hiển thị số dòng của từng bảng
print("Số dòng bảng quốc tịch:", nationality_check.count())
print("Số dòng bảng câu lạc bộ:", club_check.count())
print("Số dòng bảng vị trí:", position_check.count())

Số dòng bảng quốc tịch: 165
Số dòng bảng câu lạc bộ: 673
Số dòng bảng vị trí: 728


## TỔNG KẾT

- **Nhiệm vụ:** Xử lý dữ liệu bằng Apache Spark, xây dựng các bảng dữ liệu phụ, tối ưu hóa quá trình xử lý và đánh giá hiệu suất thực thi truy vấn.

- **Đọc và làm sạch dữ liệu:** Thiết lập kết nối giữa Spark và HDFS, đọc dữ liệu cầu thủ nam từ tệp `male_players.csv` vào Spark DataFrame (10.003.590 bản ghi, mỗi bản ghi là một cầu thủ ở một lần cập nhật của một phiên bản FIFA). Dữ liệu được chọn lọc 12 cột cần thiết, chuẩn hóa chuỗi (cắt khoảng trắng, chuyển chuỗi rỗng và giá trị như "N/A", "null" thành NULL) và kiểm tra trùng lặp theo bộ khoá `player_id`, `fifa_version`, `fifa_update`. Kết quả không phát hiện bản ghi trùng, số dòng giữ nguyên 10.003.590. Dữ liệu sạch được lưu ra HDFS dưới dạng Parquet để các bước sau đọc nhanh và tiết kiệm tài nguyên.

- **Bảng dữ liệu phụ:** Các bảng phụ được tính trên phiên bản FIFA mới nhất (`fifa_version` = [...]), mỗi cầu thủ lấy một bản ghi ở lần cập nhật cuối, nhằm tránh cộng dồn giá trị của cùng một cầu thủ qua nhiều phiên bản. Gồm ba bảng theo quốc tịch ([165] dòng), câu lạc bộ ([673] dòng) và vị trí thi đấu ([728] dòng), cung cấp số lượng cầu thủ, chỉ số tổng thể trung bình, mức lương trung bình, chỉ số tiềm năng trung bình và giá trị thị trường theo từng nhóm. Các bảng được lưu trên HDFS dưới định dạng Parquet và đã được đọc lại để kiểm tra.

- **Tối ưu hiệu suất:** Sử dụng cơ chế cache của Spark ([persist với MEMORY_ONLY]) để lưu DataFrame vào bộ nhớ khi thực hiện các thao tác lặp lại. Benchmark truy vấn tính mức lương trung bình theo câu lạc bộ cho kết quả: trước cache [...] giây, sau cache [...] giây, mức thay đổi [...]%. Vì dữ liệu đã được lưu dạng Parquet với số cột ít nên truy vấn gốc vốn đã nhanh, do đó mức cải thiện của cache nhỏ hơn so với khi đọc trực tiếp từ CSV. Sau benchmark, cache được giải phóng bằng `unpersist()`.

> Nhìn chung, nhiệm vụ Spark ETL đã thực hiện được các nội dung chính gồm đọc và xử lý dữ liệu, tạo các bảng phụ phục vụ truy vấn phân tích, lưu trữ dữ liệu tổng hợp dưới định dạng Parquet trên HDFS và đánh giá hiệu suất thông qua benchmark. Kết quả góp phần chuẩn bị dữ liệu có cấu trúc cho các bước khai thác và phân tích tiếp theo của dự án Big Data.